In [ ]:
import sys
# umap-learn 0.5.x uses check_array(ensure_all_finite=...) which was removed in
# scikit-learn 1.8.0.  Pin to <1.8 so UMAP works, then restart the kernel.
!{sys.executable} -m pip install "umap-learn>=0.5.7" "scikit-learn>=1.5,<1.8" -q
print("Done — restart the kernel before running the next cells.")


In [7]:
import sys, os, time, warnings
warnings.filterwarnings('ignore')

# metric_functions.py lives 2 directories above this notebook
_root = os.path.abspath(os.path.join(os.getcwd(), '..', '..'))
if _root not in sys.path:
    sys.path.insert(0, _root)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import StandardScaler, LabelEncoder, label_binarize
from sklearn.decomposition import PCA
from sklearn.neighbors import KNeighborsClassifier, NearestNeighbors
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.model_selection import StratifiedKFold
from sklearn.manifold import trustworthiness
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                              roc_auc_score, pairwise_distances)
from scipy.stats import spearmanr
import umap

import metric_functions as mf
# Safety patch: compute_mean_projection calls ai_projection_mean_stable
if not hasattr(mf, 'ai_projection_mean_stable'):
    mf.ai_projection_mean_stable = mf.ai_projection_mean

from metric_functions import (
    stack_npp_to_ppn,
    compute_mean_projection,
    project_stack_to_tangent_features,
    compute_pairwise_distance_matrix,
)

# ── Data loading ──────────────────────────────────────────────────────────────
DATA_DIR = "."
data = np.load(os.path.join(DATA_DIR, "covariance_matrices.npz"), allow_pickle=True)
X = data["X"]   # (4752, 22, 22)
y = data["y"]   # (4752,) string labels

meta = pd.read_csv(os.path.join(DATA_DIR, "metadata.csv"))
print(f"X shape : {X.shape}")
print(f"Classes : {np.unique(y).tolist()}")
print(f"Per class: {pd.Series(y).value_counts().sort_index().to_dict()}")

# ── Subsets ───────────────────────────────────────────────────────────────────
# small: sample_a x scales {2,6,10}  -> n=396
idx_small = ((meta["sample"] == "sample_a") & (meta["scale"].isin([2, 6, 10]))).values
X_small, y_small = X[idx_small], y[idx_small]

# sample_a only -> n=432
idx_a = (meta["sample"] == "sample_a").values
X_a, y_a = X[idx_a], y[idx_a]

# sample_a + sample_b -> n=864
idx_ab = meta["sample"].isin(["sample_a", "sample_b"]).values
X_ab, y_ab = X[idx_ab], y[idx_ab]

X_full, y_full = X, y

print(f"\nSubset sizes -- small:{len(y_small)}  sample_a:{len(y_a)}  "
      f"ab:{len(y_ab)}  full:{len(y_full)}")

METRICS     = ["bw", "ai", "loge"]
UMAP_PARAMS = dict(n_neighbors=15, min_dist=0.1, random_state=42)


X shape : (4752, 22, 22)
Classes : ['aluminium_foil', 'brown_bread', 'corduroy', 'cork', 'cotton', 'cracker', 'lettuce_leaf', 'linen', 'white_bread', 'wood', 'wool']
Per class: {'aluminium_foil': 432, 'brown_bread': 432, 'corduroy': 432, 'cork': 432, 'cotton': 432, 'cracker': 432, 'lettuce_leaf': 432, 'linen': 432, 'white_bread': 432, 'wood': 432, 'wool': 432}

Subset sizes -- small:396  sample_a:1188  ab:2376  full:4752


## Task 1 — Manifold UMAP vs Tangent-Space UMAP

Compare two UMAP strategies for each geometry (BW, AI, LogE):
- **Manifold UMAP**: run on the precomputed pairwise SPD distance matrix
- **Tangent-Space UMAP**: project to log-map vectors, run standard Euclidean UMAP

Metrics: neighborhood overlap, trustworthiness, Spearman distance correlation.

In [9]:
# =============================================================================
# TASK 1 -- Manifold UMAP vs Tangent-Space UMAP
# =============================================================================
# X_small (n=396) keeps pairwise distance computation tractable.

results_t1 = {}

for metric in METRICS:
    print(f"\n{'='*52}\n  {metric.upper()} geometry\n{'='*52}")

    # A. Manifold UMAP -- pairwise SPD distances -> precomputed UMAP
    print("  [A] Computing pairwise distance matrix ...", flush=True)
    D = compute_pairwise_distance_matrix(X_small, metric=metric, verbose=False)
    print(f"      D shape={D.shape}  max={D.max():.4f}  "
          f"symmetry_err={np.max(np.abs(D - D.T)):.2e}")

    print("  [A] Running UMAP(metric='precomputed') ...", flush=True)
    emb_manifold = umap.UMAP(metric="precomputed", **UMAP_PARAMS).fit_transform(D)

    # B. Tangent-Space UMAP -- log-map -> vectorise -> Euclidean UMAP
    print("  [B] Computing barycenter ...", flush=True)
    X_ppn   = stack_npp_to_ppn(X_small)
    M, info = compute_mean_projection(X_ppn, metric=metric)
    print(f"      converged: iters={info['iters']}  final_dist={info['final']:.2e}")

    print("  [B] Projecting to tangent space ...", flush=True)
    Z = project_stack_to_tangent_features(X_small, M, metric=metric)
    print(f"      Z shape={Z.shape}")

    print("  [B] Running UMAP(metric='euclidean') ...", flush=True)
    emb_tangent = umap.UMAP(metric="euclidean", **UMAP_PARAMS).fit_transform(Z)

    results_t1[metric] = {
        "D": D, "M": M, "Z": Z,
        "emb_manifold": emb_manifold,
        "emb_tangent":  emb_tangent,
    }
    print("  Done.")

print("\n\u2713  Task 1 computations complete.")



  BW geometry
  [A] Computing pairwise distance matrix ...
      D shape=(396, 396)  max=2.5277  symmetry_err=0.00e+00
  [A] Running UMAP(metric='precomputed') ...


TypeError: check_array() got an unexpected keyword argument 'ensure_all_finite'

In [ ]:
# Task 1 -- Plot: 3x2 UMAP grid (geometry x method) --------------------------

label_set = np.unique(y_small)
cmap      = plt.cm.tab20
colors    = {lbl: cmap(i / len(label_set)) for i, lbl in enumerate(label_set)}

fig, axes = plt.subplots(3, 2, figsize=(13, 16))
fig.suptitle(
    "Task 1 -- Manifold UMAP vs Tangent-Space UMAP\n"
    "(KTH-TIPS2, sample_a, scales 2/6/10,  n=396)",
    fontsize=13, fontweight="bold")

for row, metric in enumerate(METRICS):
    for col, (key, title) in enumerate([
            ("emb_manifold", "Manifold UMAP\n(precomputed SPD distances)"),
            ("emb_tangent",  "Tangent-Space UMAP\n(Euclidean on log-map vectors)")]):
        ax  = axes[row, col]
        emb = results_t1[metric][key]
        for lbl in label_set:
            mask = y_small == lbl
            # color= (not c=) avoids matplotlib misinterpreting an RGBA tuple
            # wrapped in a list as an array of scalar color values
            ax.scatter(emb[mask, 0], emb[mask, 1],
                       color=colors[lbl], label=lbl,
                       s=14, alpha=0.75, linewidths=0)
        ax.set_title(f"{metric.upper()} -- {title}", fontsize=10)
        ax.set_xlabel("UMAP 1", fontsize=8)
        ax.set_ylabel("UMAP 2", fontsize=8)
        ax.tick_params(labelsize=7)

handles = [
    plt.Line2D([0], [0], marker="o", color="w",
               markerfacecolor=colors[l], markersize=7, label=l)
    for l in label_set
]
fig.legend(handles=handles, loc="lower center", ncol=4, fontsize=8,
           bbox_to_anchor=(0.5, -0.01), frameon=True)

plt.tight_layout(rect=[0, 0.06, 1, 1])
plt.savefig("task1_umap_comparison.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved: task1_umap_comparison.png")


In [ ]:
# Task 1 -- Metrics: Neighborhood Overlap, Trustworthiness, Spearman rho -----

def neighborhood_overlap(D_manifold, Z_tangent, k=15):
    """Mean fraction of k-NN in manifold space also present in tangent space."""
    nn_m = NearestNeighbors(n_neighbors=k+1, metric="precomputed").fit(D_manifold)
    nn_t = NearestNeighbors(n_neighbors=k+1, metric="euclidean").fit(Z_tangent)
    _, idx_m = nn_m.kneighbors(D_manifold)
    _, idx_t = nn_t.kneighbors(Z_tangent)
    return float(np.mean([
        len(set(idx_m[i, 1:]) & set(idx_t[i, 1:])) / k
        for i in range(len(Z_tangent))
    ]))

def spearman_dist_corr(D_manifold, Z_tangent):
    """Spearman rho between upper-triangle manifold and tangent Euclidean distances."""
    n   = D_manifold.shape[0]
    tri = np.triu_indices(n, k=1)
    d_man = D_manifold[tri]
    d_tan = pairwise_distances(Z_tangent)[tri]
    rho, _ = spearmanr(d_man, d_tan)
    return float(rho)

K = 15
print(f"Task 1 -- Neighborhood Preservation Metrics  (k={K})\n")
print(f"{'Geometry':<10}  {'Nbr Overlap':>12}  {'Trustworthiness':>16}  {'Spearman rho':>12}")
print("-" * 57)

t1_metrics = {}
for metric in METRICS:
    D  = results_t1[metric]["D"]
    Z  = results_t1[metric]["Z"]
    no = neighborhood_overlap(D, Z, k=K)
    tw = trustworthiness(D, Z, n_neighbors=K, metric="precomputed")
    sp = spearman_dist_corr(D, Z)
    t1_metrics[metric] = {"nbr_overlap": no, "trustworthiness": tw, "spearman": sp}
    print(f"{metric.upper():<10}  {no:>12.4f}  {tw:>16.4f}  {sp:>12.4f}")

print("\nInterpretation:")
print("  Nbr Overlap      -- fraction of k-NN in manifold space preserved in tangent space")
print("  Trustworthiness  -- are tangent-space neighbors truly close on the manifold?")
print("  Spearman rho     -- rank correlation of all pairwise distances (manifold vs Euclidean tangent)")


## Task 2 — Raw vs Standardized Tangent Features

**Q1** Feature scale comparison across geometries (boxplots of per-feature SDs)  
**Q2** PCA cumulative variance explained — raw vs standardized  
**Q3** Classification accuracy — raw vs standardized (answered in Task 3 barplot)

In [ ]:
# =============================================================================
# TASK 2 -- Raw vs Standardized Tangent Features
# =============================================================================

# Q1: Do different geometries produce differently-scaled tangent features?
# Reuse M and Z from results_t1 (already computed on X_small).

t2_data = {}
for metric in METRICS:
    Z = results_t1[metric]["Z"]
    t2_data[metric] = {"M": results_t1[metric]["M"], "Z": Z, "sds": Z.std(axis=0)}

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
fig.suptitle("Task 2 Q1 -- Tangent Feature Standard Deviations per Geometry",
             fontsize=13, fontweight="bold")

for ax, metric in zip(axes, METRICS):
    sds = t2_data[metric]["sds"]
    ax.boxplot(sds, vert=True, patch_artist=True,
               boxprops=dict(facecolor="lightsteelblue", color="steelblue"),
               medianprops=dict(color="firebrick", linewidth=2),
               flierprops=dict(marker=".", markersize=3, alpha=0.5))
    ax.set_title(
        f"{metric.upper()}\nmedian={np.median(sds):.4f}  "
        f"IQR=[{np.percentile(sds,25):.4f}, {np.percentile(sds,75):.4f}]\n"
        f"range=[{sds.min():.2e}, {sds.max():.4f}]",
        fontsize=9)
    ax.set_ylabel("Standard Deviation")
    ax.set_xlabel("253 tangent features (pooled)")
    ax.set_xticks([])

plt.tight_layout()
plt.savefig("task2_feature_sds.png", dpi=150)
plt.show()
print("Saved: task2_feature_sds.png")


In [ ]:
# Task 2 Q2 -- Does standardisation change the PCA latent structure? ----------

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
fig.suptitle("Task 2 Q2 -- PCA Cumulative Variance Explained: Raw vs Standardized",
             fontsize=13, fontweight="bold")

n_feat  = 253
q_range = np.arange(1, n_feat + 1)
q_table = {}

for ax, metric in zip(axes, METRICS):
    Z_raw = t2_data[metric]["Z"]
    Z_std = StandardScaler().fit_transform(Z_raw)
    q_table[metric] = {}

    for Z, label, ls, col in [(Z_raw, "Raw",          "-",  "steelblue"),
                               (Z_std, "Standardized", "--", "darkorange")]:
        pca = PCA().fit(Z)
        cve = np.cumsum(pca.explained_variance_ratio_)
        ax.plot(q_range[:len(cve)], cve, ls, label=label, color=col, lw=2)
        for thresh in [0.90, 0.95]:
            idx_t = int(np.searchsorted(cve, thresh))
            if idx_t < len(cve):
                ax.plot(idx_t + 1, thresh, "o", color=col, markersize=5)
        tag = "raw" if label == "Raw" else "std"
        q_table[metric][tag] = {
            "q90": int(np.searchsorted(cve, 0.90)) + 1,
            "q95": int(np.searchsorted(cve, 0.95)) + 1,
        }

    ax.axhline(0.90, color="gray", linestyle=":", alpha=0.6)
    ax.axhline(0.95, color="gray", linestyle="-.", alpha=0.6)
    ax.set_title(f"{metric.upper()}", fontsize=11)
    ax.set_xlabel("Components q"); ax.set_ylabel("CVE(q)")
    ax.set_xlim(0, 80); ax.set_ylim(0, 1.02)
    ax.legend(fontsize=8); ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig("task2_pca_cve.png", dpi=150)
plt.show()
print("Saved: task2_pca_cve.png\n")

print("Components needed to reach 90%/95% CVE:")
print(f"{'Geometry':<8}  {'Raw@90%':>8}  {'Raw@95%':>8}  {'Std@90%':>8}  {'Std@95%':>8}")
print("-" * 48)
for metric in METRICS:
    r = q_table[metric]["raw"]; s = q_table[metric]["std"]
    print(f"{metric.upper():<8}  {r['q90']:>8}  {r['q95']:>8}  {s['q90']:>8}  {s['q95']:>8}")


## Task 3 — Classification with Dimension Reduction

Three pipelines per geometry:
- **A** Full tangent features → classifier
- **B** PCA-q → classifier
- **C** UMAP-q → classifier

Classifiers: KNN, SVM, LR. Latent dims: q ∈ {5, 10, 20}. 5-fold stratified nested CV. No data leakage.

In [ ]:
# =============================================================================
# TASK 3 -- Classification Pipelines with Nested Cross-Validation
# =============================================================================

def run_nested_cv(X_mats, y_labels, metric, pipeline="A", q=10,
                  standardize=False, classifier="knn", n_outer=5, random_state=42):
    """
    5-fold stratified nested CV with zero data leakage.
    Barycenter, scaler, and DR are all fitted inside each training fold.

    pipeline   : 'A' full features | 'B' PCA-q | 'C' UMAP-q
    classifier : 'knn' | 'svm' | 'lr'
    """
    le    = LabelEncoder()
    y_enc = le.fit_transform(y_labels)
    n_cls = len(le.classes_)
    skf   = StratifiedKFold(n_splits=n_outer, shuffle=True, random_state=random_state)

    accs, precs, recs, aucs, runtimes = [], [], [], [], []

    for train_idx, test_idx in skf.split(X_mats, y_enc):
        t0 = time.time()
        X_tr, X_te = X_mats[train_idx], X_mats[test_idx]
        y_tr, y_te = y_enc[train_idx],  y_enc[test_idx]

        # 1. Barycenter on training matrices only
        M_fold, _ = compute_mean_projection(stack_npp_to_ppn(X_tr), metric=metric)

        # 2. Tangent projection using training barycenter
        Z_tr = project_stack_to_tangent_features(X_tr, M_fold, metric=metric)
        Z_te = project_stack_to_tangent_features(X_te, M_fold, metric=metric)

        # 3. Optional standardisation -- fit on train only
        if standardize:
            sc   = StandardScaler()
            Z_tr = sc.fit_transform(Z_tr)
            Z_te = sc.transform(Z_te)

        # 4. Dimension reduction -- fit on train only
        if pipeline == "B":
            n_comp = min(q, Z_tr.shape[1], Z_tr.shape[0] - 1)
            dr     = PCA(n_components=n_comp, random_state=random_state)
            Z_tr   = dr.fit_transform(Z_tr)
            Z_te   = dr.transform(Z_te)
        elif pipeline == "C":
            dr   = umap.UMAP(n_components=q, metric="euclidean",
                             n_neighbors=min(15, len(Z_tr) - 1),
                             min_dist=0.1, random_state=random_state)
            Z_tr = dr.fit_transform(Z_tr)
            Z_te = dr.transform(Z_te)

        # 5. Classifier
        if classifier == "knn":
            clf = KNeighborsClassifier(n_neighbors=5)
        elif classifier == "svm":
            clf = SVC(kernel="rbf", C=1.0, probability=True)
        elif classifier == "lr":
            clf = LogisticRegression(max_iter=1000, C=1.0,
                                     solver="lbfgs", multi_class="multinomial")
        else:
            raise ValueError(f"Unknown classifier: {classifier}")

        clf.fit(Z_tr, y_tr)
        y_pred = clf.predict(Z_te)

        # 6. Collect metrics
        accs.append(accuracy_score(y_te, y_pred))
        precs.append(precision_score(y_te, y_pred, average="macro", zero_division=0))
        recs.append(recall_score(y_te,  y_pred, average="macro", zero_division=0))

        y_prob = (clf.predict_proba(Z_te) if hasattr(clf, "predict_proba")
                  else label_binarize(y_pred, classes=np.arange(n_cls)))
        try:
            auc_val = roc_auc_score(y_te, y_prob, multi_class="ovr",
                                    average="macro", labels=np.arange(n_cls))
        except ValueError:
            auc_val = float("nan")
        aucs.append(auc_val)
        runtimes.append(time.time() - t0)

    return {
        "accuracy":  np.mean(accs),
        "acc_std":   np.std(accs),
        "precision": np.mean(precs),
        "recall":    np.mean(recs),
        "auc":       np.nanmean(aucs),
        "runtime_s": np.sum(runtimes),
    }

print("run_nested_cv() defined.")


In [ ]:
# Task 3 -- Run full comparison table ----------------------------------------
# Default: X_small (n=396).  Swap to X_a / X_ab for more thorough results.
X_use, y_use = X_small, y_small

Q_VALUES    = [5, 10, 20]
CLASSIFIERS = ["knn", "svm", "lr"]

rows  = []
total = len(METRICS) * 2 * len(CLASSIFIERS) * (1 + len(Q_VALUES) * 2)
run_n = 0

for metric in METRICS:
    for standardize in [False, True]:
        for clf_name in CLASSIFIERS:

            # Pipeline A -- full tangent features
            run_n += 1
            print(f"[{run_n:03d}/{total}] {metric.upper()} std={standardize} "
                  f"{clf_name} A-Full", end="  ", flush=True)
            t = time.time()
            r = run_nested_cv(X_use, y_use, metric=metric, pipeline="A",
                              standardize=standardize, classifier=clf_name)
            print(f"acc={r['accuracy']:.3f}  ({time.time()-t:.1f}s)")
            rows.append({"geometry": metric.upper(), "pipeline": "A-Full",
                         "q": None, "standardize": standardize,
                         "classifier": clf_name, **r})

            for q in Q_VALUES:
                for pipe, pname in [("B", "B-PCA"), ("C", "C-UMAP")]:
                    run_n += 1
                    print(f"[{run_n:03d}/{total}] {metric.upper()} std={standardize} "
                          f"{clf_name} {pname} q={q}", end="  ", flush=True)
                    t = time.time()
                    r = run_nested_cv(X_use, y_use, metric=metric, pipeline=pipe, q=q,
                                      standardize=standardize, classifier=clf_name)
                    print(f"acc={r['accuracy']:.3f}  ({time.time()-t:.1f}s)")
                    rows.append({"geometry": metric.upper(), "pipeline": pname,
                                 "q": q, "standardize": standardize,
                                 "classifier": clf_name, **r})

df_results = pd.DataFrame(rows)
df_results = df_results.sort_values(
    ["geometry", "classifier", "pipeline", "q"]).reset_index(drop=True)
df_results.to_csv("task3_results.csv", index=False)

print("\n\n-- Task 3 Results --")
display_cols = ["geometry", "pipeline", "q", "standardize", "classifier",
                "accuracy", "acc_std", "precision", "recall", "auc", "runtime_s"]
print(df_results[display_cols].to_string(index=False,
      float_format=lambda x: f"{x:.4f}" if isinstance(x, float) else str(x)))
print("\nSaved: task3_results.csv")


In [ ]:
# Task 2 Q3 + Task 3 -- Grouped accuracy barplot (raw vs standardized) -------

for clf_name in CLASSIFIERS:
    df_clf = df_results[df_results["classifier"] == clf_name].copy()
    df_clf = df_clf[df_clf["pipeline"] != "A-Full"].copy()
    df_clf["label"] = df_clf.apply(
        lambda r: f"{r['pipeline']} q={int(r['q'])}", axis=1)

    label_order = [f"{pipe} q={q}"
                   for pipe in ["B-PCA", "C-UMAP"]
                   for q in Q_VALUES]

    fig, axes = plt.subplots(1, 3, figsize=(17, 5), sharey=True)
    fig.suptitle(
        f"Task 2 Q3 + Task 3 -- CV Accuracy [{clf_name.upper()}]: Raw vs Standardized",
        fontsize=12, fontweight="bold")

    for ax, metric in zip(axes, [m.upper() for m in METRICS]):
        sub = df_clf[df_clf["geometry"] == metric]
        x   = np.arange(len(label_order))
        w   = 0.38
        raw_accs, std_accs, raw_errs, std_errs = [], [], [], []
        for lbl in label_order:
            r = sub[(sub["label"] == lbl) & (~sub["standardize"])]
            raw_accs.append(r["accuracy"].values[0] if len(r) else 0)
            raw_errs.append(r["acc_std"].values[0]  if len(r) else 0)
            s = sub[(sub["label"] == lbl) & (sub["standardize"])]
            std_accs.append(s["accuracy"].values[0] if len(s) else 0)
            std_errs.append(s["acc_std"].values[0]  if len(s) else 0)

        ax.bar(x - w/2, raw_accs, w, yerr=raw_errs, capsize=3,
               label="Raw",          color="steelblue",  alpha=0.85)
        ax.bar(x + w/2, std_accs, w, yerr=std_errs, capsize=3,
               label="Standardized", color="darkorange",  alpha=0.85)
        ax.set_xticks(x)
        ax.set_xticklabels(label_order, rotation=40, ha="right", fontsize=8)
        ax.set_title(f"{metric}", fontsize=11)
        ax.set_ylabel("CV Accuracy (mean +/- std)")
        ax.set_ylim(0, 1.05)
        ax.legend(fontsize=8)
        ax.grid(True, axis="y", alpha=0.3)

    plt.tight_layout()
    fname = f"task3_accuracy_barplot_{clf_name}.png"
    plt.savefig(fname, dpi=150, bbox_inches="tight")
    plt.show()
    print(f"Saved: {fname}")

# -- Best configuration per geometry -----------------------------------------
print("\n-- Best configuration per geometry (highest mean accuracy) --")
print(f"{'Geometry':<8}  {'Pipeline':<10}  {'q':>4}  {'Std':>5}  {'Clf':<5}  {'Acc':>6}  {'+-':>6}")
print("-" * 55)
for metric in [m.upper() for m in METRICS]:
    sub  = df_results[df_results["geometry"] == metric]
    best = sub.loc[sub["accuracy"].idxmax()]
    q_val = best["q"]
    q_str = "--" if (q_val is None or
                     (isinstance(q_val, float) and np.isnan(q_val))) else str(int(q_val))
    print(f"{metric:<8}  {best['pipeline']:<10}  {q_str:>4}  "
          f"{str(best['standardize']):>5}  {best['classifier']:<5}  "
          f"{best['accuracy']:>6.4f}  {best['acc_std']:>6.4f}")
